# OC Robotique -  Protocoles de communication


Bienvenue dans ce tutoriel dans lequel vous allez implémenter certaines briques d'un protocole de communication.

!!! success Découvrir le tutoriel

Passer à la suite en exécutant cette cellule.

!!!



## Structure générale



!!! info Cahier des charges 
* Destinataire & émetteur
* Numéro de séquence, Ack, Timeout 
* Checksum (pour compléter 16 bit hardware CRC )
* Encryption (AES)  – Bonus

!!!


<style>
  .fancy-table {
    border-collapse: collapse;
    width: 100%;
    font-family: Arial, Helvetica, sans-serif;
    font-size: 14px;
  }
  .fancy-table thead th,
  .fancy-table thead td {
    text-align: center;                 /* Centre l'en-tête */
    background: linear-gradient(#f7e9f7,#fad7fc);
    padding: 10px 12px;
    border: 2px solid #450b66;          /* Bordure plus visible pour l'en-tête */
    color: #450b66;
    font-weight: 600;
  }
  .fancy-table tbody th,
  .fancy-table tbody td {
    padding: 10px 12px;
    border: 1px solid #cbd5e0;          /* Séparations visibles entre cellules */
    text-align: center;
    vertical-align: middle;
    background: #ffffff;
  }
  .fancy-table tbody tr:nth-child(even) td {
    background: #f7fbff;                /* Légère alternance de lignes */
  }
  /* Optionnel : effet au survol */
  .fancy-table tbody tr:hover td {
    background: #eef6ff;
  }
</style>

<table class="fancy-table">
  <thead>
    <tr>
      <td colspan="6">Messages</td>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>Destinataire</th>
      <th>Expéditeur</th>
      <th>Numéro de séquence</th>
      <th>Type de message</th>
      <th>Payload chiffrée</th>
      <th>Somme de contrôle</th>
    </tr>
    <!-- Exemple de ligne de données -->
    <tr>
      <td>0</td>
      <td>1</td>
      <td>42</td>
      <td>12 (aruco_position)</td>
      <td>[5 623 212 40]</td>
      <td>935</td>
    </tr>
    <tr>
      <td>1</td>
      <td>0</td>
      <td>42</td>
      <td>255 (ack)</td>
      <td>[  ]</td>
      <td>298</td>
    </tr>
    <tr>
      <td>0</td>
      <td>1</td>
      <td>43</td>
      <td>12 (aruco_position)</td>
      <td>[5 628 210 41]</td>
      <td>939</td>
    </tr>
    <tr>
      <td>1</td>
      <td>0</td>
      <td>43</td>
      <td>255 (ack)</td>
      <td>[  ]</td>
      <td>299</td>
    </tr>
  </tbody>
</table>




!!! info Vocabulaire :
* **Payload** = contenu brut à transmettre
* **Message** = objet qui contient la payload et les autres paramètres (destinataires, exp., sequence, etc.)
* **Trame** = suite de bytes qui contient tout le message
!!!


!!! success Suite

Passer à la suite en exécutant cette cellule.

!!!

## 0) Format de la payload

!!! info Payload
Le but de notre message est d’envoyer d’une carte A à une carte B les éléments suivants : 

* id (`int`)
* x_center (`int`)
* y_center (`int`)
* angle (`int`)

!!!
!!! note Exercice

* Créer une liste contenant des valeurs entières que vous aurez choisies pour ces 4 grandeurs.
* Utilisez les fonctions `radio.send_int(list_of_int)` et `radio.receive_int()` pour envoyer et recevoir votre payload.

!!!


In [ ]:
from microbit_radio_simu import *

# Fonctions
def envoi_message(payload):
    trame = payload
    radio.send_int(trame) # Fonction d'envoi du microbit
    
def reception_message():
    trame = radio.receive_int() # Fonction de réception du microbit
    if trame :
        payload = trame[:]
        return payload

# Expéditeur
data = [5, 623, 212, 40]
envoi_message(data)


# Destinataire
payload = reception_message()
print("Message reçu : ", payload)

payload = reception_message()
print("Message reçu : ", payload)

## 1) Catégorie de message

!!! info Catégorie
Afin de pouvoir gérer plusieurs types de messages en parallèle (par exemple la position des marqueurs, les accusés de réception, etc.), on a besoin d'identifier la catégorie à laquelle chaque message appartient. Celui-ci sera représenté par un entier de 0 à 255.

!!!
!!! note Exercice 

Complétez les fonctions suivantes en réutilisant votre programme précédent et en ajoutant à la trame l'identifiant de la catégorie de message.

!!!


In [ ]:
# Fonctions
def envoi_message(id_category, payload):
    trame = [id_category] + payload
    radio.send_int(trame)
    
def reception_message():
    trame = radio.receive_int()
    
    if trame :
        id_category = trame[0]
        payload = trame[1:]

        return id_category, payload

# Expéditeur
data = [5, 623, 212, 40]
id_aruco_msg = 12
envoi_message(id_aruco_msg, data)


# Destinataire
id_category, payload = reception_message()
print("Message type", id_category, "Contenu : ", payload)

## 2) Destinataire et expéditeur

!!! info 


!!!
!!! note Exercice 

...

!!!


In [ ]:
def envoi_message(id_dest, id_exped, id_category, payload):
    trame = [id_dest, id_exped, id_category] + payload # On ajoute destinataire et expéditeur
    radio.send_int(trame)
    
def reception_message(mon_id):
    trame = radio.receive_int()
    
    if trame :
        id_dest = trame[0] # On ajoute destinataire et expéditeur
        id_exped = trame[1]
        id_category = trame[2]
        payload = trame[3:]

        if mon_id == id_dest: # vérification du destinataire
            return id_dest, id_exped, id_category, payload
        else:
            return None,None, None, None

# Expéditeur
data = [5, 623, 212, 40]
id_aruco_msg = 12
exp_id = 0 # On ajoute destinataire et expéditeur
dest_id  = 1
envoi_message(dest_id, exp_id, id_aruco_msg, data)


# Destinataire
id_dest, id_exped, id_category, payload = reception_message(dest_id)

print("Message pour", id_dest,"de",  id_exped, "-- type", id_category, "-- Contenu :", payload)

## 3) Acquittement & Ré-envoi

!!! info 


!!!
!!! note Exercice 

...

!!!


In [ ]:
seqNum = 0
def envoi_message(id_dest, id_exped, id_category, payload):
    trame = [id_dest, id_exped, seqNum, id_category] + payload # On ajoute le numéro de séquence
    radio.send_int(trame)

def check_last_message_ack(id_exped):
    # Receive ack
    id_dest_ack, id_exped_ack, id_category_ack, _, received_seqNum = reception_message(id_exped)
    
    # Check ack
    global seqNum
    if received_seqNum == seqNum and id_category_ack == 255:
        seqNum = seqNum + 1
        return True
    return False
    
def reception_message(mon_id):
    trame = radio.receive_int()
    
    if trame :
        id_dest = trame[0]
        id_exped = trame[1]
        received_seqNum = trame[2] # On ajoute le numéro de séquence
        id_category = trame[3]
        payload = trame[4:]

        if mon_id == id_dest:
            # On envoie un accusé si le message n'en était pas un
            if id_category != 255:
                envoi_message(id_exped, id_dest, 255, [])                
            return id_dest, id_exped, id_category, payload, received_seqNum
        
    return None, None, None, None, None

# Expéditeur
data = [5, 623, 212, 40]
id_aruco_msg = 12
exp_id = 0
dest_id  = 1
envoi_message(dest_id, exp_id, id_aruco_msg, data)


# Destinataire
id_dest, id_exped, id_category, payload, _ = reception_message(dest_id)
print("Message pour", id_dest,"de",  id_exped, "-- type", id_category, "-- Contenu :", payload)


# Expéditeur
print("Message acked : ", check_last_message_ack(exp_id))

!!! tip Retry

À noter que dans un protocole complet, en cas de non-réception d'un accusé de réception après un certain temps, on **renverrait le message N fois jusqu'à réception**.

Ici, comme on est à la fois l'expéditeur et le destinataire et que les messages sont tous reçus, on va passer cette étape

!!!


## 4) Somme de contrôle


!!! info 


!!!
!!! note Exercice 

...

!!!


In [ ]:
seqNum = 0
def envoi_message(id_dest, id_exped, id_category, payload):
    trame = [id_dest, id_exped, seqNum, id_category] + payload  # On ajoute une somme de contrôle
    checksum = sum(trame)
    trame = trame + [checksum]
    radio.send_int(trame)

def check_last_message_ack(id_exped):
    # Receive ack
    id_dest_ack, id_exped_ack, id_category_ack, _, received_seqNum = reception_message(id_exped)
    
    # Check ack
    global seqNum
    if received_seqNum == seqNum and id_category_ack == 255:
        seqNum = seqNum + 1
        return True
    return False
    
def reception_message(mon_id):
    trame = radio.receive_int()
    
    if trame :
        id_dest = trame[0]
        id_exped = trame[1]
        received_seqNum = trame[2]
        id_category = trame[3]
        payload = trame[4:-1]
        checksum =  trame[-1] # On ajoute une somme de contrôle

        if mon_id == id_dest and checksum == sum(trame[:-1]) : # On recalcule et compare la somme de contrôle
            if id_category != 255:
                envoi_message(id_exped, id_dest, 255, [])                
            return id_dest, id_exped, id_category, payload, received_seqNum
        
    return None, None, None, None, None

# Expéditeur
data = [5, 623, 212, 40]
id_aruco_msg = 12
exp_id = 0
dest_id  = 1
envoi_message(dest_id, exp_id, id_aruco_msg, data)


# Destinataire
id_dest, id_exped, id_category, payload, _ = reception_message(dest_id)
print("Message pour", id_dest,"de",  id_exped, "-- type", id_category, "-- Contenu :", payload)


# Expéditeur
print("Message acked", check_last_message_ack(exp_id))